In [ ]:
import pandas as pd
import sys
import re
!{sys.executable} -m pip install openpyxl

In [ ]:
archivo = "C:\\Users\\User\\Downloads\\ALGO 25\\1.AlbertoLineroGo Bic Ene Info Contable.xlsx"

df = pd.read_excel(
    archivo,
    sheet_name="BCE PRUEBA CC"
)

In [ ]:
pd.read_excel(archivo,
    sheet_name="BCE PRUEBA CC")

In [ ]:
# Leer encabezado con mes y año
enc = pd.read_excel(archivo, nrows=4, header=None, sheet_name="BCE PRUEBA CC")
texto = str(enc.iloc[3, 0])

# Expresión regular para "De <mes> <año>"
patron = r"De\s+([a-záéíóúñ]+)\s+(\d{4})"
m = re.search(patron, texto, flags=re.IGNORECASE)

if m:
    mes = m.group(1).capitalize()
    anio = int(m.group(2))
else:
    mes, anio = None, None

# Añadir columnas nuevas a tu DataFrame (en esa fila)
enc.loc[3, "Mes"] = mes
enc.loc[3, "Año"] = anio

print(enc.head())

In [ ]:
# 2) Leer tabla real (con encabezados correctos)
df = pd.read_excel(archivo, header=4, sheet_name="BCE PRUEBA CC")

# 3) Añadir columnas
df["Año"] = anio
df["Mes"] = mes

In [ ]:
df.tail()

In [ ]:
# 2. Cambiar tipos de columnas (equivalente a Table.TransformColumnTypes)
df = df.astype({
    "Año": "int64",
    "Mes": "string",
    "Nivel": "string",
    "Transaccional": "string",
    "Código cuenta contable": "Int64",   # <- nullable int
    "Nombre cuenta contable": "string",
    "Código centro costo": "string",
    "Nombre centro costo": "string",
    "Saldo inicial": "Int64",            # si puede tener vacíos
    "Movimiento débito": "float",
    "Movimiento crédito": "float",
    "Saldo final": "float"
})



In [ ]:
# 3. Quitar columnas Nivel y Transaccional
df = df.drop(columns=["Nivel", "Transaccional"])

In [ ]:
# 4. Dividir columna "Código centro costo" por "-"
df[["Código centro costo.1", "Código centro costo.2"]] = df["Código centro costo"].str.split("-", n=1, expand=True)


In [ ]:
# 5. Convertir a numéricos
df["Código centro costo.1"] = pd.to_numeric(df["Código centro costo.1"], errors="coerce")
df["Código centro costo.2"] = pd.to_numeric(df["Código centro costo.2"], errors="coerce")


In [ ]:
# 6. Combinar Código centro costo.2 + Nombre centro costo → "Subcuenta"
df["Subcuenta"] = df["Código centro costo.2"].astype("Int64").astype(str) + " " + df["Nombre centro costo"]


In [ ]:
# 7. Eliminar "Código centro costo.1"
df = df.drop(columns=["Código centro costo.1", "Código centro costo", "Nombre centro costo"])


In [ ]:
# 8. Ajustar tipos
df = df.astype({
    "Código cuenta contable": "string",
    "Saldo inicial": "float",
    "Movimiento débito": "float",
    "Movimiento crédito": "float",
    "Saldo final": "float"
})

In [ ]:
# 9. Agregar columna "Largo" = longitud de "Código cuenta contable"
df["Largo"] = df["Código cuenta contable"].str.len()

In [ ]:
# 10. Agregar columna "Cuenta" con condiciones
df["Cuenta"] = df["Código cuenta contable"].str[0].map(
    {"4": "Ingreso", "5": "Gasto"}
).fillna("Costo")

In [ ]:
# 11. Agregar columna "Dato Financiero"
df["Dato Financiero"] = df.apply(
    lambda x: (
        x["Movimiento crédito"] - x["Movimiento débito"]
        if str(x["Código cuenta contable"]).startswith("4")
        else x["Movimiento débito"] - x["Movimiento crédito"]
    ),
    axis=1
)


In [ ]:
# 12. Agregar columna "Puc & Descripción"
df["Puc & Descripción"] = df["Código cuenta contable"] + " " + df["Nombre cuenta contable"]


In [ ]:
# 13. Reordenar columnas como en Power Query
df = df[[
    "Año", "Mes", "Largo", "Cuenta", "Subcuenta",
    "Código cuenta contable", "Nombre cuenta contable", "Puc & Descripción",
    "Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final", "Dato Financiero"
]]


In [ ]:
df = df[df["Código cuenta contable"].notna() & (df["Código cuenta contable"] != "")]

In [ ]:
# Obtener posición después de "Cuenta"
pos = df.columns.get_loc("Cuenta") + 1

# Insertar columnas vacías en el orden deseado
df.insert(pos, "id_EEFF", None)
df.insert(pos+1, "NombreCuenta", None)
df.insert(pos+2, "CuentaFin", None)


In [ ]:
# Exportar a Excel
df.to_excel(
    r"C:\Users\User\Downloads\limpiado_CC.xlsx",  # Ruta de salida
    index=False                                      # No guardar índice
)